# Free GPU Coding-LLM Server on Kaggle (2x T4)

Hosts a strong open coding model on Kaggle's free **2x Tesla T4 (16GB + 16GB)** GPUs using **vLLM**, and exposes an **OpenAI-compatible API** over a free Cloudflare tunnel — so you can point Cline / Continue / aider / any OpenAI-compatible tool at it from your own machine.

## Before you run
1. **Settings (right sidebar):** Accelerator = `GPU T4 x2`, Internet = `On`, Persistence = `No persistence`.
2. Run all cells top to bottom. The tunnel URL is printed by the **TUNNEL** cell.
3. Kaggle gives ~30 GPU hours/week and kills sessions after ~9–12h. Re-run when it dies — you get a new URL each time.

## Honest performance expectations
- T4 is a 2018 GPU: no bfloat16, no FlashAttention-2. We run `--dtype half`.
- Single request: expect roughly **30–60 tok/s** with the MoE model, ~20–35 tok/s with the dense 14B.
- **100+ tok/s total** is achievable across **parallel requests** (vLLM continuous batching), not on one stream.

In [ ]:
# ============ CONFIG ============
# Primary: MoE coder — only ~3B params active per token, so it's FAST on old GPUs
# while having 30B-class coding quality. 4-bit quant fits across 2x16GB with TP=2.
MODEL_ID = "cpatonn/Qwen3-Coder-30B-A3B-Instruct-AWQ"

# FALLBACK (official Qwen quant, guaranteed to work on T4 with vLLM):
# MODEL_ID = "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ"

# If the primary repo name 404s, search Hugging Face for
# "Qwen3-Coder-30B-A3B-Instruct AWQ" or "... GPTQ-Int4" and paste the repo id here.

MAX_MODEL_LEN = 32768   # context length; lower to 16384 if you hit KV-cache OOM
SERVED_NAME = "coder"    # the model name clients will use
print("Model:", MODEL_ID)

In [ ]:
# ============ INSTALL (takes a few minutes) ============
%pip install -q -U vllm huggingface_hub

# cloudflared binary for the free quick tunnel (no account needed)
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared
print("install done")

In [ ]:
# ============ START vLLM SERVER (background) ============
import subprocess, time, urllib.request, os

cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--served-model-name", SERVED_NAME,
    "--tensor-parallel-size", "2",      # split across both T4s
    "--dtype", "half",                  # T4 has no bf16
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", "0.93",
    "--host", "0.0.0.0", "--port", "8000",
    "--api-key", "kaggle-local-key",    # simple auth so random people can't use your GPU
    "--enforce-eager",                  # saves VRAM on T4; remove for a bit more speed if it fits
]
log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print("vLLM starting (model download + load can take 5-15 min on first run)...")

# wait until the server answers
for i in range(240):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
        print("vLLM is UP")
        break
    except Exception:
        if server.poll() is not None:
            print("SERVER DIED — last 40 log lines:")
            os.system("tail -40 /kaggle/working/vllm.log")
            raise SystemExit("vLLM failed to start. Try the fallback MODEL_ID or lower MAX_MODEL_LEN.")
        time.sleep(5)
else:
    raise SystemExit("Timed out waiting for vLLM. Check /kaggle/working/vllm.log")

In [ ]:
# ============ TUNNEL — prints your public URL ============
import subprocess, re, time

tun = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
start = time.time()
while time.time() - start < 60:
    line = tun.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        public_url = m.group(0)
        break

print("=" * 60)
print("OPENAI-COMPATIBLE BASE URL:", (public_url or "NOT FOUND — rerun this cell") + "/v1")
print("API KEY: kaggle-local-key")
print("MODEL NAME:", SERVED_NAME)
print("=" * 60)

In [ ]:
# ============ SMOKE TEST + SPEED CHECK ============
import urllib.request, json, time

req = urllib.request.Request(
    "http://127.0.0.1:8000/v1/chat/completions",
    data=json.dumps({
        "model": SERVED_NAME,
        "messages": [{"role": "user", "content": "Write a Python function that checks if a string is a palindrome."}],
        "max_tokens": 256,
    }).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer kaggle-local-key"},
)
t0 = time.time()
resp = json.load(urllib.request.urlopen(req, timeout=300))
dt = time.time() - t0
out_tokens = resp["usage"]["completion_tokens"]
print(resp["choices"][0]["message"]["content"][:500])
print(f"\n--- {out_tokens} tokens in {dt:.1f}s = {out_tokens/dt:.1f} tok/s (single stream) ---")

In [ ]:
# ============ KEEP ALIVE — leave this cell running ============
import time, urllib.request
while True:
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5)
        print(time.strftime("%H:%M:%S"), "alive")
    except Exception as e:
        print(time.strftime("%H:%M:%S"), "server not responding:", e)
    time.sleep(300)